# Regression Tutorial - DeepTab v2.0

This notebook demonstrates how to train regression models with DeepTab for predicting continuous targets.

**Topics covered:**
- Basic regression workflow
- Target preprocessing strategies
- Customization with configs
- Hyperparameter tuning
- Residual analysis and feature importance
- Multiple architectures comparison

**Requirements:**
```bash
pip install deeptab scikit-learn pandas numpy matplotlib scipy
```

## 1. Setup and Data Generation

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

from deeptab.models import MambularRegressor

# Set random seed for reproducibility
np.random.seed(42)

# Generate synthetic data
n_samples, n_features = 1000, 5
X = np.random.randn(n_samples, n_features)
coefficients = np.random.randn(n_features)
y = np.dot(X, coefficients) + np.random.randn(n_samples)

# Create DataFrame
df = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(n_features)])
df["target"] = y

print(f"Dataset shape: {df.shape}")
print(f"Target statistics:")
print(f"  Mean: {y.mean():.3f}")
print(f"  Std: {y.std():.3f}")
print(f"  Min: {y.min():.3f}")
print(f"  Max: {y.max():.3f}")

## 2. Train/Test Split

In [ ]:
X = df.drop(columns=["target"])
y = df["target"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

## 3. Train with Default Settings

DeepTab automatically handles preprocessing and validation splitting.

In [ ]:
# Instantiate and train
model = MambularRegressor()
model.fit(X_train, y_train, max_epochs=50)

## 4. Evaluate and Predict

In [ ]:
# Evaluate on test set
metrics = model.evaluate(X_test, y_test)
print("Test Metrics:")
print(f"  RMSE: {metrics['rmse']:.3f}")
print(f"  MAE: {metrics['mae']:.3f}")
print(f"  R² score: {model.score(X_test, y_test):.3f}")

# Get predictions
predictions = model.predict(X_test)
print(f"\nPredictions shape: {predictions.shape}")
print(f"Sample predictions: {predictions[:10]}")
print(f"Prediction mean: {predictions.mean():.3f}")

## 5. Customization with Configs

Use PreprocessingConfig and TrainerConfig for fine-grained control.

In [ ]:
from deeptab.configs import MambularConfig, PreprocessingConfig, TrainerConfig

# Model architecture
model_cfg = MambularConfig(
    d_model=256,
    n_layers=8,
    dropout=0.2,
)

# Preprocessing
prep_cfg = PreprocessingConfig(
    numerical_preprocessing="quantile",  # Transform to uniform distribution
    use_ple=True,                         # Piecewise Linear Encoding
    n_bins=50,
)

# Training
trainer_cfg = TrainerConfig(
    lr=5e-4,
    batch_size=256,
    max_epochs=150,
    patience=20,
    lr_scheduler="cosine",
    optimizer="adamw",
    weight_decay=1e-4,
)

# Create and train custom model
model_custom = MambularRegressor(
    model_config=model_cfg,
    preprocessing_config=prep_cfg,
    trainer_config=trainer_cfg,
)

model_custom.fit(X_train, y_train, max_epochs=150)

# Evaluate
metrics_custom = model_custom.evaluate(X_test, y_test)
print(f"Custom Model RMSE: {metrics_custom['rmse']:.3f}")
print(f"Custom Model R²: {model_custom.score(X_test, y_test):.3f}")

## 6. Target Preprocessing

For skewed targets, log transformation often helps.

In [ ]:
# Generate positive skewed target
y_positive = np.abs(y) + 1.0

# Log transform
y_log = np.log1p(y_positive)  # log(1 + y)

# Split
X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)

# Train on log-transformed target
model_log = MambularRegressor()
model_log.fit(X_train_log, y_train_log, max_epochs=50)

# Predict and transform back
predictions_log = model_log.predict(X_test_log)
predictions_original = np.expm1(predictions_log)  # exp(y) - 1

# Evaluate on original scale
y_test_positive = y_positive[X_test_log.index]
rmse = np.sqrt(mean_squared_error(y_test_positive, predictions_original))
r2 = r2_score(y_test_positive, predictions_original)

print(f"Log-transform model RMSE (original scale): {rmse:.3f}")
print(f"Log-transform model R² (original scale): {r2:.3f}")

## 7. Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "model_config__d_model": [128, 256],
    "model_config__n_layers": [4, 6, 8],
    "trainer_config__lr": [5e-4, 1e-3],
    "preprocessing_config__numerical_preprocessing": ["standard", "quantile"],
}

model_grid = MambularRegressor()

grid_search = GridSearchCV(
    model_grid,
    param_grid,
    cv=3,
    scoring="neg_mean_squared_error",
    n_jobs=1,
    verbose=2,
)

print("Running grid search...")
grid_search.fit(X_train, y_train)

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best CV RMSE: {np.sqrt(-grid_search.best_score_):.3f}")

# Test set performance
best_model = grid_search.best_estimator_
test_r2 = best_model.score(X_test, y_test)
print(f"Test R²: {test_r2:.3f}")

## 8. Cross-Validation

In [ ]:
from sklearn.model_selection import cross_val_score

model_cv = MambularRegressor()

# Negative MSE (sklearn convention)
scores = cross_val_score(
    model_cv, X_train, y_train,
    cv=5,
    scoring="neg_mean_squared_error",
)

rmse_scores = np.sqrt(-scores)
print(f"CV RMSE scores: {rmse_scores}")
print(f"Mean RMSE: {rmse_scores.mean():.3f} (+/- {rmse_scores.std():.3f})")

## 9. Residual Analysis

In [ ]:
import matplotlib.pyplot as plt
from scipy import stats

# Get predictions
predictions = model.predict(X_test)
residuals = y_test - predictions

# Create plots
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Residual plot
axes[0].scatter(predictions, residuals, alpha=0.5)
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Residuals")
axes[0].set_title("Residual Plot")

# Q-Q plot
stats.probplot(residuals, dist="norm", plot=axes[1])
axes[1].set_title("Q-Q Plot")

plt.tight_layout()
plt.show()

# Statistics
print(f"Mean residual: {residuals.mean():.4f}")
print(f"Std residual: {residuals.std():.4f}")

## 10. Feature Importance

In [ ]:
from sklearn.inspection import permutation_importance

# Compute permutation importance
result = permutation_importance(
    model, X_test, y_test,
    n_repeats=10,
    random_state=42,
    scoring="neg_mean_squared_error",
)

# Create DataFrame
importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": result.importances_mean,
    "std": result.importances_std,
}).sort_values("importance", ascending=False)

print(importance_df)

# Plot
plt.figure(figsize=(8, 6))
plt.barh(importance_df["feature"], importance_df["importance"])
plt.xlabel("Permutation Importance")
plt.title("Feature Importance")
plt.tight_layout()
plt.show()

## 11. Comparing Different Architectures

In [ ]:
from deeptab.models import (
    FTTransformerRegressor,
    ResNetRegressor,
    NODERegressor,
    MambularRegressor,
)

architectures = [
    FTTransformerRegressor,
    ResNetRegressor,
    NODERegressor,
    MambularRegressor,
]

results = []
for ModelClass in architectures:
    print(f"\nTraining {ModelClass.__name__}...")
    model = ModelClass()
    model.fit(X_train, y_train, max_epochs=50)
    r2 = model.score(X_test, y_test)
    results.append((ModelClass.__name__, r2))
    print(f"  R² = {r2:.3f}")

# Display results
print("\n" + "="*50)
print("Architecture Comparison")
print("="*50)
for name, r2 in sorted(results, key=lambda x: x[1], reverse=True):
    print(f"{name:30s}: R² = {r2:.3f}")

## 12. Mixed Data Types

In [ ]:
# Create dataset with numerical and categorical features
df_mixed = pd.DataFrame({
    "age": np.random.randint(18, 80, size=1000),
    "income": np.random.randint(20000, 200000, size=1000),
    "city": np.random.choice(["NYC", "LA", "Chicago"], size=1000),
    "education": np.random.choice(["HS", "BS", "MS", "PhD"], size=1000),
    "experience_years": np.random.randint(0, 40, size=1000),
    "target": np.random.randn(1000) * 10000 + 50000,
})

X_mixed = df_mixed.drop(columns=["target"])
y_mixed = df_mixed["target"].values

X_train_mixed, X_test_mixed, y_train_mixed, y_test_mixed = train_test_split(
    X_mixed, y_mixed, test_size=0.2, random_state=42
)

# Train - automatically handles both numerical and categorical
model_mixed = MambularRegressor()
model_mixed.fit(X_train_mixed, y_train_mixed, max_epochs=50)

metrics_mixed = model_mixed.evaluate(X_test_mixed, y_test_mixed)
print(f"Mixed data R²: {model_mixed.score(X_test_mixed, y_test_mixed):.3f}")
print(f"Mixed data RMSE: {metrics_mixed['rmse']:.3f}")

## 13. Save and Load

In [ ]:
# Save model
model.save("regressor_model.pkl")
print("Model saved!")

# Load model
from deeptab.models import MambularRegressor
loaded_model = MambularRegressor.load("regressor_model.pkl")

# Use loaded model
predictions_loaded = loaded_model.predict(X_test)
print(f"Loaded model R²: {loaded_model.score(X_test, y_test):.3f}")

## Summary

In this tutorial, you learned how to:
- ✅ Train regression models with DeepTab v2.0
- ✅ Customize preprocessing and training with configs
- ✅ Handle target preprocessing (log transform, standardization)
- ✅ Perform hyperparameter tuning and cross-validation
- ✅ Analyze residuals and feature importance
- ✅ Compare different model architectures
- ✅ Work with mixed data types

**Next steps:**
- Try [Classification Tutorial](classification.ipynb) for categorical targets
- Explore [Distributional Regression](distributional.ipynb) for uncertainty quantification
- Check [Experimental Models](experimental.ipynb) for cutting-edge architectures

**Documentation:** https://deeptab.readthedocs.io/